# Bocetos gratis de personajes con FLUX.1 schnell

Genera imágenes de prueba de un personaje **sin gastar créditos de Higgsfield**, en la GPU gratuita de Google Colab.
Sirve para **definir** el personaje (silueta, colores, ropa, actitud). La versión final se produce después en Higgsfield.

- Modelo: **FLUX.1 [schnell]** de Black Forest Labs, licencia **Apache-2.0** (permite uso comercial).
- Va comprimido a 4 bits (NF4) para que entre en la GPU T4 gratuita.
- Solo personajes **originales**: nada de personajes existentes ni personas reales que no hayan dado permiso.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.

> No probado todavía en una sesión real de Colab (2026-11-07). Si algo falla, anota el error y pásaselo a Claude.


In [ ]:
# 1. Instalar (tarda 1-2 minutos)
!pip -q install -U diffusers transformers accelerate bitsandbytes sentencepiece protobuf
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cargar el modelo
**Antes, una sola vez (recomendado):** el modelo oficial está «restringido» en Hugging Face.
1. Crea una cuenta gratis en https://huggingface.co y entra a https://huggingface.co/black-forest-labs/FLUX.1-schnell → acepta las condiciones (*Agree and access repository*).
2. Crea una llave de lectura en https://huggingface.co/settings/tokens (*Create new token* → tipo **Read**).
3. Aquí en Colab: ícono de la **llave** (🔑 *Secretos*) en la barra izquierda → *Agregar secreto* → nombre `HF_TOKEN`, valor: tu llave → activa *Acceso del notebook*.
   **No pegues la llave en el código ni en ningún chat.**

Si no hay llave, la celda intenta con una copia comprimida de la comunidad, que no pide registro (sin probar).
La primera vez tarda varios minutos. Si Colab se reinicia por falta de RAM, reinicia y vuelve a correr las celdas 1 y 2.

In [ ]:
import torch, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DBnb
from transformers import T5EncoderModel, BitsAndBytesConfig as TBnb

# 1) Llave de Hugging Face desde los Secretos de Colab (si existe)
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if token:
    from huggingface_hub import login
    login(token=token)
    print("Llave de Hugging Face cargada desde Secretos")
else:
    print("Sin llave HF_TOKEN: se intentará la copia de la comunidad")

dtype = torch.float16  # la T4 no tiene bfloat16 rápido
pipe = None

# 2) Modelo oficial (necesita llave y haber aceptado las condiciones)
if token:
    try:
        REPO = "black-forest-labs/FLUX.1-schnell"
        transformer = FluxTransformer2DModel.from_pretrained(
            REPO, subfolder="transformer", torch_dtype=dtype,
            quantization_config=DBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype))
        text_encoder_2 = T5EncoderModel.from_pretrained(
            REPO, subfolder="text_encoder_2", torch_dtype=dtype,
            quantization_config=TBnb(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dtype))
        pipe = FluxPipeline.from_pretrained(REPO, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=dtype)
        print("Modelo oficial listo")
    except Exception as e:
        print("No se pudo cargar el modelo oficial:", type(e).__name__, str(e)[:300])

# 3) Copia comprimida de la comunidad (sin registro; sin probar)
if pipe is None:
    try:
        pipe = FluxPipeline.from_pretrained("vftfnn/flux1-schnell-nf4", torch_dtype=dtype)
        print("Copia de la comunidad lista")
    except Exception as e:
        raise RuntimeError("No se pudo cargar ningún modelo. Configura la llave HF_TOKEN (ver arriba) "
                           "y vuelve a correr esta celda. Detalle: " + type(e).__name__ + ": " + str(e)[:300])

pipe.enable_model_cpu_offload()
gc.collect(); torch.cuda.empty_cache()
print("Modelo listo")

## 3. Pegar el prompt y generar
Pega el **prompt** que entrega el afinador de personajes (`creador-personajes/afinado/<personaje>.md`).

- `SEMILLAS`: cada número es una variante. **Repetir la misma semilla** con un prompt cambiado muestra solo el efecto
  del cambio (así se compara A contra B).
- Formato vertical 9:16 (576 × 1024). Cada imagen tarda unos segundos a un minuto en la T4.

In [ ]:
GENERAR_SPIKE = False  # ponlo en True para generar el ejemplo de Spike
if GENERAR_SPIKE:
    PERSONAJE = "spike"
    PROMPT = """Realistic photo of an original, wildly exaggerated eccentric character named Spike, larger-than-life comedic caricature proportions: absurdly tall and stick-thin with extra-long arms, his signature hair is a giant crown of rigid punk liberty spikes, about a dozen spikes each as long as his forearm, entirely glowing NEON ORANGE, fluorescent safety-orange, ultra saturated, perfectly stiff and glossy. Bright green expressive eyes, thick bleached eyebrows, a wide cheeky full-toothed grin, chewing bubblegum. Oversized shiny purple velvet tracksuit with enormous padded shoulders, a fluffy neon-yellow feather boa, chunky rings on every finger, huge white platform sneakers, dramatic rock-star pose. Seamless white infinity studio, soft even light, full body, vertical 9:16."""
    SEMILLAS = [1, 2, 3, 4]
    
    import os, torch
    from IPython.display import display
    os.makedirs(f"/content/bocetos/{PERSONAJE}", exist_ok=True)
    imagenes = []
    for s in SEMILLAS:
        img = pipe(PROMPT, width=576, height=1024, num_inference_steps=4, guidance_scale=0.0,
                   max_sequence_length=256, generator=torch.Generator("cpu").manual_seed(s)).images[0]
        ruta = f"/content/bocetos/{PERSONAJE}/{PERSONAJE}_semilla{s}.png"
        img.save(ruta); imagenes.append(img); print("Guardada:", ruta)
    
    # Mostrar en fila
    from PIL import Image
    fila = Image.new("RGB", (288 * len(imagenes), 512), "white")
    for i, im in enumerate(imagenes): fila.paste(im.resize((288, 512)), (288 * i, 0))
    display(fila)

## 3b. Tiky y Lumi v2 (3D contra 2D) — ya cargado
Genera las 6 variantes con las mismas 2 semillas (12 imágenes). Son versiones **cortas** de los prompts de la v2
(`afinado/pollito.md` y `afinado/vaca.md`) para no pasar el límite de texto del modelo. Tarda unos 10 a 15 minutos en la T4.
Si solo quieres un personaje, borra las líneas que no quieras de `VARIANTES`.

In [ ]:
TRES_D = "Modern 3D cartoon render of a soft matte vinyl toy figure, children's animation style, simple rounded shapes, clean bright colors."
DOS_D = "Modern 2D flat cartoon illustration for children, clean vector shapes, thick dark outlines, flat bright colors, simple soft shading."
TIKY = ("An original adorable baby chick shaped like a smooth upright egg, no neck, tiny stubby wings and legs. "
        "A small discreet tuft of three short thick rounded flame-shaped feathers on top of his head, neon orange. "
        "Warm sunflower yellow body, pale cream belly, big round dark-brown eyes with a sparkle, small coral-peach beak open in a big amazed smile, rosy cheeks, coral-peach feet. "
        "Leaning forward on tiptoe, one wing raised as if asking a question. Plain pastel sky-blue background, single full-body pose, centered, vertical.")
LUMI = ("An original adorable gentle cow shaped like a big smooth upright egg, no neck, short stubby legs and arms. "
        "{CRAYON}Mostly creamy white with two large irregular black patches with wavy edges, one across her back and belly, one on top of her head. "
        "Black lower legs like socks, lilac hooves, small cream horns, floppy ears, big honey-brown eyes with a sparkle, calm knowing smile, wide soft lilac muzzle. "
        "Tapping her chin with one hoof, looking up as if she had a bright idea. Plain pastel sky-blue background, single full-body pose, centered, vertical.")
CRAYON = "A short chunky lilac crayon tucked behind her right ear. "

VARIANTES = {
    "tiky_3d": f"{TRES_D} {TIKY}",
    "tiky_2d": f"{DOS_D} {TIKY}",
    "lumi_crayon_3d": f"{TRES_D} {LUMI.format(CRAYON=CRAYON)}",
    "lumi_crayon_2d": f"{DOS_D} {LUMI.format(CRAYON=CRAYON)}",
    "lumi_sin_crayon_3d": f"{TRES_D} {LUMI.format(CRAYON='')}",
    "lumi_sin_crayon_2d": f"{DOS_D} {LUMI.format(CRAYON='')}",
}
SEMILLAS = [1, 2]

import os, torch
from PIL import Image
from IPython.display import display
os.makedirs("/content/bocetos/tiky_lumi_v2", exist_ok=True)
tok = pipe.tokenizer_2
for nombre, prompt in VARIANTES.items():
    n = len(tok(prompt).input_ids)
    print(f"{nombre}: {n} tokens" + ("  ⚠️ se corta (máx. 256)" if n > 256 else ""))
    fila = []
    for s in SEMILLAS:
        img = pipe(prompt, width=576, height=1024, num_inference_steps=4, guidance_scale=0.0,
                   max_sequence_length=256, generator=torch.Generator("cpu").manual_seed(s)).images[0]
        img.save(f"/content/bocetos/tiky_lumi_v2/{nombre}_semilla{s}.png"); fila.append(img)
    tira = Image.new("RGB", (288 * len(fila), 512), "white")
    for i, im in enumerate(fila): tira.paste(im.resize((288, 512)), (288 * i, 0))
    print(nombre); display(tira)
print("Listo: /content/bocetos/tiky_lumi_v2")

## 4. Descargar
Baja las imágenes y muéstraselas a Claude con tu veredicto («me gusta la semilla 3, pero…»). El afinador anota qué
semilla y qué prompt funcionaron.

In [ ]:
!cd /content && zip -qr bocetos.zip bocetos
from google.colab import files
files.download("/content/bocetos.zip")

## 5. (Experimental) Probar movimiento con Wan 2.2
Wan 2.2 TI2V-5B (Apache-2.0) puede animar una imagen, pero en la T4 gratuita queda **muy justo de memoria** (las
fuentes no coinciden). Si quieres intentarlo, hay un cuaderno de la comunidad: https://github.com/theelderemo/wan2.2-google-colab
(elige el modo TI2V-5B, baja la resolución y la cantidad de cuadros si se queda sin memoria). Si no funciona, las
pruebas de movimiento se hacen en Higgsfield con Seedance 2.0 Mini en 480p (6 créditos por 12 s).